# Chess Stats Tutorial

This notebook walks through how to ingest your chess.com game history and explore it using the `chess_stats` query API.

**Workflow:**
1. Run the ingest script to pull game history from chess.com → stored as `data/games.parquet`
2. Load a `GameDB` and query / visualize the data
3. (Optional) Join with external data (health metrics, sleep logs, etc.) for correlation analysis

## 1. Ingest game history

Run this once (and again periodically to pick up new games). It is safe to re-run — already-fetched months are skipped.

In [ ]:
import subprocess
result = subprocess.run(
    ["uv", "run", "python", "-m", "chess_stats.ingest", "--username", "nirum"],
    capture_output=True, text=True, cwd=".."
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

## 2. Connect to the local data

In [ ]:
import sys
sys.path.insert(0, "..")

import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from chess_stats import GameDB

db = GameDB("../data")

## 3. Summary statistics

In [ ]:
summary = db.summary()
for k, v in summary.items():
    print(f"{k:25s}: {v}")

## 4. Raw game data

`db.games()` returns a polars DataFrame with one row per game. All standard polars operations apply.

In [ ]:
games = db.games()
print(f"Shape: {games.shape}")
games.head(5)

In [ ]:
# Filter to rapid games only
rapid = db.games(time_class="rapid")
print(f"Rapid games: {len(rapid)}")
rapid.head(3)

## 5. Rating over time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

for ax, tc in zip(axes, ["rapid", "blitz"]):
    hist = db.rating_history(time_class=tc)
    if hist.is_empty():
        ax.set_title(f"{tc.title()} (no data)")
        continue
    dates = hist["end_datetime"].to_list()
    ratings = hist["my_rating"].to_list()
    ax.plot(dates, ratings, linewidth=0.8, alpha=0.7)
    ax.set_title(f"{tc.title()} rating")
    ax.set_xlabel("Date")
    ax.set_ylabel("Rating")
    # Show only a few x-tick labels to avoid crowding
    ax.xaxis.set_major_locator(mticker.MaxNLocator(5))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

fig.tight_layout()
plt.show()

## 6. Win rate by hour of day

This is the core hook for correlating performance with external factors. Hours are converted from UTC to Pacific time (America/Los_Angeles, DST-aware).

In [ ]:
from zoneinfo import ZoneInfo
from datetime import datetime, timezone

def utc_hour_to_pacific(utc_hour: int) -> int:
    """Convert a UTC hour (0-23) to Pacific hour using today's DST offset."""
    pacific = ZoneInfo("America/Los_Angeles")
    dt = datetime.now(tz=timezone.utc).replace(hour=utc_hour, minute=0, second=0, microsecond=0)
    return dt.astimezone(pacific).hour

hourly = db.performance_by_hour()
hourly = (
    hourly
    .with_columns(
        pl.col("hour_of_day")
        .map_elements(utc_hour_to_pacific, return_dtype=pl.Int32)
        .alias("hour_pacific")
    )
    .sort("hour_pacific")
)
hourly

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

hours = hourly["hour_pacific"].to_list()
win_rates = hourly["win_rate"].to_list()
n_games = hourly["n_games"].to_list()

bars = ax.bar(hours, win_rates, color="steelblue", alpha=0.8)

# Annotate with game counts
for bar, n in zip(bars, n_games):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        str(n), ha="center", va="bottom", fontsize=7
    )

ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, label="50% win rate")
ax.set_xlabel("Hour of day (Pacific)")
ax.set_ylabel("Win rate")
ax.set_title("Win rate by hour of day (Pacific time)")
ax.set_xticks(range(24))
ax.legend()
fig.tight_layout()
plt.show()

## 7. Win rate by day of week

In [ ]:
daily = db.performance_by_day()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(daily["day_name"].to_list(), daily["win_rate"].to_list(), color="coral", alpha=0.85)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax.set_ylabel("Win rate")
ax.set_title("Win rate by day of week")
fig.tight_layout()
plt.show()

## 8. Opening analysis

In [ ]:
openings = db.opening_stats(min_games=5)
print("Top openings by games played:")
openings.head(10)

In [ ]:
# Best and worst openings (min 10 games for reliability)
reliable = openings.filter(pl.col("n_games") >= 10)

print("Best openings (by win rate):")
print(reliable.sort("win_rate", descending=True).head(5))
print("\nWorst openings (by win rate):")
print(reliable.sort("win_rate").head(5))

## 9. Joining with external data

The `games` DataFrame has an `end_datetime` column (ISO 8601, UTC). You can join any time-indexed external dataset — health metrics, sleep data from Apple Health / Oura / Garmin, etc.

Below is a template assuming you have a health DataFrame with a `date` column (YYYY-MM-DD).

In [ ]:
import polars as pl

games = db.games()

# Extract the date in Pacific time for a day-level join
games_with_date = games.with_columns(
    pl.col("end_datetime")
    .str.to_datetime(time_unit="us", time_zone="UTC")
    .dt.convert_time_zone("America/Los_Angeles")
    .dt.date()
    .cast(pl.Utf8)
    .alias("date")
)

# --- Replace this with your actual health data ---
# health = pl.read_csv("health_export.csv")  # columns: date, hrv, sleep_hours, ...
# joined = games_with_date.join(health, on="date", how="left")
# -------------------------------------------------

# Example: compute win rate for the top half vs bottom half of any metric
# metric_col = "hrv"
# median_val = joined[metric_col].median()
# (
#     joined
#     .with_columns((pl.col(metric_col) > median_val).alias("high_metric"))
#     .group_by("high_metric")
#     .agg(
#         pl.len().alias("n_games"),
#         ((pl.col("result") == "win").sum() / pl.len()).alias("win_rate")
#     )
# )

print("Template ready. Load your health data and uncomment the join above.")
games_with_date.select(["game_id", "date", "result", "my_rating", "hour_of_day"]).head(5)